# Brain Tumor Classification from MRI images

The model aims to classify brain tumors from MRI images. The dataset contains the images of Glioma, Meningioma, Pituitary tumor, and the normal condition. This is done by implementing Convolutional Neural Network (CNN) with PyTorch.

[Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)

Reference: Msoud Nickparvar. (2026). Brain Tumor MRI Dataset [Data set]. Kaggle. https://doi.org/10.34740/KAGGLE/DSV/14832123

## 1. Downloading and unzipping the dataset

In [14]:
! export KAGGLE_API_TOKEN=xxxx #Please enter your Kaggle API Token: https://github.com/Kaggle/kaggle-cli/tree/main/docs

In [15]:
#!/bin/bash
! kaggle datasets download masoudnickparvar/brain-tumor-mri-dataset

Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
brain-tumor-mri-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
! unzip /content/brain-tumor-mri-dataset.zip

Archive:  /content/brain-tumor-mri-dataset.zip
replace Testing/glioma/Te-gl_1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

##2. Importing necessary libraries

In [ ]:
#Reference: https://www.youtube.com/watch?v=ZBfpkepdZlw
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import os
from scipy import misc
import glob
from PIL import Image
from torchvision import datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.optim as optim
import torch.nn.functional as F

##3. Setting the device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

##4. Preparing dataloaders

In [ ]:
#loading data
train_data_dir="/content/Training"

test_data_dir="/content/Testing"
#Reference: https://stackoverflow.com/questions/10439104/reading-bmp-files-in-python



In [ ]:
img_size=32
#Reference: https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
train_transform = transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

tv_transform = transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
#Reference: https://stackoverflow.com/questions/69199273/torchvision-imagefolder-could-not-find-any-class-folder

train_data=datasets.ImageFolder(train_data_dir, transform=train_transform)
test_data=datasets.ImageFolder(test_data_dir,transform=tv_transform)

#128 arbitrary
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)




##5. Constructing the neural network

In [ ]:
#Reference: https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        #Reference: https://www.youtube.com/watch?v=CtzfbUwrYGI
        self.conv1 = nn.Conv2d(3, 6, 5) #for the case of 256x256 #(256-5)/1 +1=252 (6,252,252)
        self.pool = nn.MaxPool2d(2, 2) #(6,126,126)
        self.conv2 = nn.Conv2d(6, 16, 5) #(126-5)/1+1 =122 (16,122,122) ->(16,61,61) ->Flatten(16*61*61)
        self.fc1 = nn.Linear(16 * 5* 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 4)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()
net.to(device)

##6. Setting the loss function and optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum = 0.9)


##7. Training the model

In [10]:
for epoch in range(10):
  print(f"Training epoch {epoch}...")

  running_loss =0.0
   #Reference: https://zenn.dev/zzz/articles/970e5df5cf1f65
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data[0].to(device), data[1].to(device)

    optimizer.zero_grad()

    outputs = net(inputs)
    loss = criterion(outputs, labels)
    #print("loss:",loss)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  print((f'Loss:{running_loss / len(train_loader):.4f}'))

print("finished Training")


Training epoch 0...
Loss:1.3046
Training epoch 1...
Loss:1.0276
Training epoch 2...
Loss:0.7219
Training epoch 3...
Loss:0.5406
Training epoch 4...
Loss:0.4454
Training epoch 5...
Loss:0.3602
Training epoch 6...
Loss:0.3106
Training epoch 7...
Loss:0.2501
Training epoch 8...
Loss:0.2027
Training epoch 9...
Loss:0.1821
finished Training


##8. Testing the model

In [11]:
torch.save(net.state_dict(), 'trained_net.pth')

In [12]:
net = Net()
net.to(device)
net.load_state_dict(torch.load("trained_net.pth"))

<All keys matched successfully>

In [13]:
#Reference: https://www.youtube.com/watch?v=CtzfbUwrYGI
correct = 0
total = 0
net.eval()

with torch.no_grad():
  for data in test_loader:
    images,labels = data[0].to(device), data[1].to(device)
    outputs=net(images)
    _, predicted = torch.max(outputs,1)
    total+=labels.size(0)
    correct+=(predicted == labels).sum().item()

accuracy = round(100*correct/total,2)

print(f"Accuracy: {accuracy}%")


Accuracy: 85.62%
